# Classification evaluation

Predict whether an approved order will arrive after its estimated delivery date. Training uses a 180-day historical window with a 45-day buffer. The notebook compares classifiers on a random stratified development split and evaluates monthly predictions using ranking, probability and daily-capacity measures. The months shown here were inspected during development, so their results are not an independent final test.

## 1. Setup and data

One row represents an approved order. Only information available by its scoring date may enter model features.
The models use 31 base order-level predictors plus four seller and product history predictors computed from outcomes known by each order’s approval day. A separate feature check compares this set with the original 31 predictors.


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from src import LOOKBACK, PXD, RANDOM_STATE
from src.data import load_olist_data
from src.features import (build_feature_table, add_historical_performance,
                          NUM_COLS, HISTORY_NUM_COLS, CAT_COLS, LEAKAGE_COLS)
from src.labels import get_required_dates, labels_as_of, attach_prediction_labels
from src.evaluation import classification_metrics
from classification_eval import extra_metrics, paired_monthly_ap

pd.set_option('display.max_columns', 30)
olist = load_olist_data(verbose=False)
orders = olist['orders']
base_features = build_feature_table(olist)
numeric_cols = NUM_COLS + HISTORY_NUM_COLS
feature_cols = numeric_cols + CAT_COLS
features = add_historical_performance(base_features, olist)
assert not set(feature_cols) & set(LEAKAGE_COLS)
assert features['order_id'].is_unique
print(f'{len(features):,} approved-order feature rows')

98,959 approved-order feature rows


## 2. Labels and the historical window

The 45-day gap gives delivery outcomes time to mature. `labels_as_of` counts an overdue, undelivered order as late; it excludes outcomes unavailable by the run date. We check label availability rather than assuming the gap resolves every order.

In [2]:
def training_window(run_date, feature_table=None):
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    table = features if feature_table is None else feature_table
    rows = table.loc[table['order_approved_dt'].between(start, end)].copy()
    rows = labels_as_of(rows, run_date)
    known = rows.loc[rows['label_as_of_run'].notna()].copy()
    known['late'] = known['label_as_of_run'].astype(int)
    return rows, known

RUN_DATE = '2018-06-02'
window, labelled = training_window(RUN_DATE)
display(window['label_status'].value_counts().rename_axis('status').to_frame('orders'))
print(f'Known labels: {len(labelled):,}/{len(window):,} ({len(labelled)/len(window):.2%})')
assert labelled['order_approved_dt'].max() < pd.Timestamp(RUN_DATE) - pd.Timedelta(days=LOOKBACK)

,orders
status,
on_time_delivered,34593
late_delivered,4213
late_undelivered,1150
invalid_or_missing_dates,2
unresolved,2


Known labels: 39,956/39,960 (99.99%)


## 3. Random stratified development split

Reserve a stratified portion of the historical window for validation. Its results can guide model development; a later untouched sample is needed for final evaluation.

In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(
    labelled[feature_cols], labelled['late'], test_size=30/PXD,
    random_state=RANDOM_STATE, stratify=labelled['late'])
print(f'Train: {len(X_train):,}; validation: {len(X_valid):,}; late rate: {y_valid.mean():.2%}')
# Keep validation outcomes out of every fitted history feature.
held_out_ids = labelled.loc[X_valid.index, 'order_id']
fold_features = add_historical_performance(
    base_features, olist, exclude_order_ids=held_out_ids)
X_train = fold_features.loc[X_train.index, feature_cols]
X_valid = fold_features.loc[X_valid.index, feature_cols]


Train: 33,296; validation: 6,660; late rate: 13.42%


## 4. Classification pipelines

Each pipeline fits preprocessing on its own training rows. Scores of 0.5 or above receive the late flag.

In [4]:
def prepare_catboost(X):
    data = X.copy()
    data[numeric_cols] = data[numeric_cols].astype(float)
    for column in CAT_COLS:
        data[column] = data[column].astype('string').fillna('Missing').astype(str)
    return data


def models_for(y):
    ratio = y.eq(0).sum() / y.eq(1).sum()
    plain = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_cols),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ], sparse_threshold=0)
    scaled = ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('scaler', StandardScaler())]), numeric_cols),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])
    return {
        'Logistic Regression': Pipeline([('preprocessor', scaled),
            ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000,
                                               random_state=RANDOM_STATE))]),
        'XGBoost': Pipeline([('preprocessor', clone(plain)),
            ('classifier', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                         scale_pos_weight=ratio, random_state=RANDOM_STATE,
                                         eval_metric='logloss'))]),
        'Random Forest': Pipeline([('preprocessor', clone(plain)),
            ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   class_weight='balanced', random_state=RANDOM_STATE,
                                                   n_jobs=-1))]),
        'CatBoost': Pipeline([('preprocessor', FunctionTransformer(prepare_catboost,
                                                                   validate=False)),
            ('classifier', CatBoostClassifier(cat_features=CAT_COLS, iterations=100,
                                               scale_pos_weight=ratio, random_seed=RANDOM_STATE,
                                               verbose=False, allow_writing_files=False))]),
        'LightGBM': Pipeline([('preprocessor', clone(plain)),
            ('classifier', LGBMClassifier(scale_pos_weight=ratio, random_state=RANDOM_STATE,
                                          verbosity=-1, n_jobs=-1))]),
    }

models = models_for(y_train)
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Fitted {name}')

Fitted Logistic Regression
Fitted XGBoost
Fitted Random Forest
Fitted CatBoost
Fitted LightGBM


## 4b. Hyperparameter tuning with time-ordered cross-validation

The pipelines above use fixed settings (for example 100 trees and depth 6). This section searches a small grid for each model and scores every candidate on the same expanding-window folds inside the training window.

- **Folds are time-ordered.** Each fold trains on earlier approval days and validates on the next block of days, so a candidate is never scored on orders older than the ones it was fitted on. No approval day is split across train and validation.
- **History features are rebuilt per fold.** The four seller and product history predictors are recomputed with that fold's validation orders excluded, the same rule used for the development split and the monthly backtests. They are built once per fold and reused by every candidate.
- **Selection metric is average precision**, which suits the rarer late class and does not depend on the 0.5 threshold. Daily top-10% coverage is recorded alongside it. Class weights stay fixed at the training-fold ratio, so only the model settings change.
- **Compare gains with fold-to-fold spread.** A tuned score that beats the default by less than the fold standard deviation is not evidence of a real improvement.
- **The monthly backtests are not an independent test of these settings.** The grid is searched on the 2018-06-02 training window only, so settings chosen here have seen data from before the April and May run dates. Treat tuned backtest results as development evidence, and keep a later month untouched for a final check.

In [5]:
import time
from sklearn.metrics import average_precision_score
from sklearn.model_selection import ParameterGrid, ParameterSampler
from src.splits import day_blocked_time_series_folds
from classification_eval import daily_capacity

N_CV_FOLDS = 4   # expanding-window folds inside the training window
GAP_DAYS = 0     # days left out between a fold's training and validation days

tune_data = labelled.sort_values('order_approved_dt', kind='stable').reset_index(drop=True)
cv_folds = day_blocked_time_series_folds(
    tune_data['order_approved_dt'], n_splits=N_CV_FOLDS, gap_days=GAP_DAYS)

cv_data = []
for k, (train_idx, valid_idx) in enumerate(cv_folds, 1):
    train_rows, valid_rows = tune_data.iloc[train_idx], tune_data.iloc[valid_idx]
    # Rebuild the history features without this fold's validation outcomes.
    fold_table = add_historical_performance(
        base_features, olist, exclude_order_ids=valid_rows['order_id']).set_index('order_id')
    cv_data.append({
        'X_train': fold_table.loc[train_rows['order_id'], feature_cols],
        'y_train': train_rows['late'].to_numpy(),
        'X_valid': fold_table.loc[valid_rows['order_id'], feature_cols],
        'y_valid': valid_rows['late'].to_numpy(),
        'valid_meta': valid_rows[['order_approved_dt', 'late']].rename(
            columns={'order_approved_dt': 'approval_date', 'late': 'actual_label'}),
    })
    print(f'Fold {k}: train {len(train_rows):,} orders '
          f'({train_rows["order_approved_dt"].min():%Y-%m-%d} to {train_rows["order_approved_dt"].max():%Y-%m-%d}) '
          f'-> validate {len(valid_rows):,} orders '
          f'({valid_rows["order_approved_dt"].min():%Y-%m-%d} to {valid_rows["order_approved_dt"].max():%Y-%m-%d}); '
          f'late rate {valid_rows["late"].mean():.2%}')
    assert train_rows['order_approved_dt'].max() < valid_rows['order_approved_dt'].min()

Fold 1: train 6,545 orders (2017-10-20 to 2017-11-24) -> validate 8,298 orders (2017-11-25 to 2017-12-30); late rate 13.18%
Fold 2: train 14,843 orders (2017-10-20 to 2017-12-30) -> validate 8,124 orders (2017-12-31 to 2018-02-04); late rate 8.09%
Fold 3: train 22,967 orders (2017-10-20 to 2018-02-04) -> validate 8,719 orders (2018-02-05 to 2018-03-12); late rate 20.70%
Fold 4: train 31,686 orders (2017-10-20 to 2018-03-12) -> validate 8,270 orders (2018-03-13 to 2018-04-17); late rate 12.67%


In [6]:
# Small grids keep run time reasonable; widen them once the feature set is frozen.
SEARCH_SPACES = {
    'Logistic Regression': {'C': list(np.logspace(-3, 1, 9))},
    'Random Forest': {'n_estimators': [100, 200, 300], 'max_depth': [6, 10, 14],
                      'min_samples_leaf': [5, 20, 50], 'max_features': ['sqrt', 0.3, 0.5]},
    'XGBoost': {'n_estimators': [100, 200, 400], 'max_depth': [3, 4, 6, 8],
                'learning_rate': [0.02, 0.05, 0.1], 'subsample': [0.7, 0.85, 1.0],
                'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_weight': [1, 5, 20]},
    'CatBoost': {'iterations': [100, 200, 400], 'depth': [4, 6, 8],
                 'learning_rate': [0.03, 0.06, 0.1], 'l2_leaf_reg': [1, 3, 10]},
    'LightGBM': {'n_estimators': [100, 200, 400], 'learning_rate': [0.02, 0.05, 0.1],
                 'num_leaves': [15, 31, 63], 'min_child_samples': [20, 50, 100],
                 'subsample': [0.7, 1.0], 'subsample_freq': [1],
                 'colsample_bytree': [0.6, 0.8, 1.0]},
}
N_ITER = {'Logistic Regression': 9, 'Random Forest': 8, 'XGBoost': 12,
          'CatBoost': 8, 'LightGBM': 12}


def with_params(model, params):
    """Return `model` with `params` applied to its classifier step."""
    return model.set_params(**{f'classifier__{key}': value for key, value in params.items()})


def cv_evaluate(name, params):
    """Fit one candidate on every fold; return per-fold average precision and top-10% coverage."""
    average_precision, coverage = [], []
    for fold in cv_data:
        model = with_params(models_for(pd.Series(fold['y_train']))[name], params)
        model.fit(fold['X_train'], fold['y_train'])
        score = model.predict_proba(fold['X_valid'])[:, list(model.classes_).index(1)]
        average_precision.append(average_precision_score(fold['y_valid'], score))
        coverage.append(daily_capacity(fold['valid_meta'].assign(score=score))['coverage'])
    return average_precision, coverage


cv_rows = []
for name, space in SEARCH_SPACES.items():
    started = time.time()
    n_iter = min(N_ITER[name], len(ParameterGrid(space)))
    candidates = [{}] + list(ParameterSampler(space, n_iter=n_iter, random_state=RANDOM_STATE))
    for number, params in enumerate(candidates):
        average_precision, coverage = cv_evaluate(name, params)
        cv_rows.append({'model': name, 'is_default': number == 0, 'params': params,
                        'ap_mean': np.mean(average_precision), 'ap_std': np.std(average_precision),
                        'coverage_mean': np.mean(coverage), 'fold_ap': average_precision})
    print(f'{name}: {len(candidates) - 1} candidates + default in {time.time() - started:.0f}s')
cv_results = pd.DataFrame(cv_rows)

Logistic Regression: 9 candidates + default in 8s
Random Forest: 8 candidates + default in 22s
XGBoost: 12 candidates + default in 25s
CatBoost: 8 candidates + default in 42s
LightGBM: 12 candidates + default in 78s


In [7]:
def best_row(group):
    return group.loc[group['ap_mean'].idxmax()]


summary_rows, tuned_params = [], {}
for name, group in cv_results.groupby('model', sort=False):
    default = group.loc[group['is_default']].iloc[0]
    best = best_row(group)
    tuned_params[name] = best['params']
    summary_rows.append({
        'model': name,
        'default_ap': default['ap_mean'], 'default_ap_std': default['ap_std'],
        'tuned_ap': best['ap_mean'], 'tuned_ap_std': best['ap_std'],
        'gain': best['ap_mean'] - default['ap_mean'],
        'gain_exceeds_fold_std': best['ap_mean'] - default['ap_mean'] > best['ap_std'],
        'default_coverage': default['coverage_mean'], 'tuned_coverage': best['coverage_mean'],
        'best_params': best['params'],
    })
tuning_summary = pd.DataFrame(summary_rows).set_index('model')
with pd.option_context('display.max_colwidth', None, 'display.width', 200):
    display(tuning_summary.round(4))


def tuned_models_for(y, params=None):
    """Same pipelines as `models_for`, with the selected hyperparameters applied."""
    params = tuned_params if params is None else params
    models = models_for(y)
    return {name: with_params(model, params.get(name, {})) for name, model in models.items()}

,default_ap,default_ap_std,tuned_ap,tuned_ap_std,gain,gain_exceeds_fold_std,default_coverage,tuned_coverage,best_params
model,,,,,,,,,
Logistic Regression,0.2337,0.0706,0.2350,0.0708,0.0013,False,0.2293,0.2270,{'C': 10.0}
Random Forest,0.2858,0.0588,0.2919,0.0652,0.0061,False,0.2258,0.2316,"{'n_estimators': 200, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 6}"
XGBoost,0.2794,0.0619,0.2877,0.0585,0.0083,False,0.2301,0.2340,"{'subsample': 0.85, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.02, 'colsample_bytree': 0.6}"
CatBoost,0.2615,0.0629,0.3034,0.0655,0.0419,False,0.2146,0.2543,"{'learning_rate': 0.03, 'l2_leaf_reg': 1, 'iterations': 100, 'depth': 4}"
LightGBM,0.2758,0.0550,0.2908,0.0638,0.0150,False,0.2215,0.2359,"{'subsample_freq': 1, 'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 20, 'learning_rate': 0.02, 'colsample_bytree': 0.6}"


The table compares each model's default settings with the best candidate on the same folds. `gain_exceeds_fold_std` flags gains larger than the fold standard deviation of the tuned score. Where it is False, keep the default. `tuned_models_for(y)` builds the same pipelines as `models_for(y)` with the selected settings, so it can replace `models_for` in the validation and backtest sections once the tuned results have been reviewed. The equal-average ensemble is not tuned separately.

## 5. Development validation metrics

Accuracy measures overall correctness. Precision and recall describe the late-order decisions, while F1 combines them. ROC-AUC and average precision evaluate ranking across thresholds; average precision is especially useful for the less common late class. Brier score measures probability error (lower is better). The daily-capacity measure below assesses how many late orders a fixed contact budget could reach. No threshold tuning or probability calibration is fitted here.

In [8]:
validation_rows = labelled.loc[X_valid.index, ['order_id', 'order_approved_dt', 'late']].copy()
validation_rows = validation_rows.rename(columns={'order_approved_dt': 'approval_date',
                                                   'late': 'actual_label'})
validation_results = []
for name, model in models.items():
    score = model.predict_proba(X_valid)[:, list(model.classes_).index(1)]
    prediction = validation_rows.copy()
    prediction['score'] = score
    result = classification_metrics(y_valid, (score >= 0.5).astype(int), score)
    result.update(extra_metrics(prediction))
    validation_results.append({'model': name, **result})
validation_table = pd.DataFrame(validation_results).set_index('model')
display(validation_table[['accuracy', 'precision', 'recall', 'f1_score',
                          'roc_auc', 'average_precision', 'brier_score',
                          'coverage', 'lift']].round(3))

,accuracy,precision,recall,f1_score,roc_auc,average_precision,brier_score,coverage,lift
model,,,,,,,,,
Logistic Regression,0.687,0.235,0.592,0.336,0.690,0.270,0.217,0.260,2.341
XGBoost,0.751,0.293,0.605,0.394,0.753,0.407,0.174,0.287,2.594
Random Forest,0.726,0.265,0.589,0.366,0.733,0.383,0.189,0.281,2.533
CatBoost,0.731,0.272,0.596,0.373,0.733,0.383,0.178,0.260,2.341
LightGBM,0.741,0.285,0.616,0.390,0.754,0.405,0.174,0.296,2.674


## 6. Monthly development backtests

Each monthly backtest fits every model once on the second calendar day of the month, using the eligible historical window whose labels are mature by that run date. For example, the April 2 fit scores orders approved on April 1, then remains fixed for orders approved during the rest of April. The notebook calculates those daily scores in one batch after the fact; no April orders are used to fit the April model.

An order already delivered by the end of its approval day is not scored. Outcomes are attached at each order’s approval date plus 45 days. Unknown outcomes remain in the scored population and daily capacity; label-based metrics use only known outcomes and report their availability.

Each monthly fit uses the training portion of a random stratified split of its historical window.

In [9]:
def monthly_predictions(run_date):
    month = pd.Period(run_date, freq='M')
    month_ids = base_features.loc[
        base_features['order_approved_dt'].dt.to_period('M').eq(month), 'order_id']
    month_features = add_historical_performance(
        base_features, olist, exclude_order_ids=month_ids)
    _, history = training_window(run_date, month_features)
    X_fit, _, y_fit, _ = train_test_split(
        history[feature_cols], history['late'], test_size=30/PXD,
        random_state=RANDOM_STATE, stratify=history['late'])
    cohort = month_features.loc[
        month_features['order_approved_dt'].dt.to_period('M').eq(month)].copy()
    delivered_day = cohort['order_delivered_customer_date'].dt.normalize()
    cohort = cohort.loc[delivered_day.isna() |
                        delivered_day.gt(cohort['order_approved_dt'])].copy()
    if cohort.empty:
        return pd.DataFrame()
    base = cohort[['order_id', 'order_approved_dt']].rename(
        columns={'order_approved_dt': 'approval_date'})
    fitted = models_for(y_fit)
    scores = {}
    for name, model in fitted.items():
        model.fit(X_fit, y_fit)
        scores[name] = model.predict_proba(cohort[feature_cols])[:,
                          list(model.classes_).index(1)]
    scores['Equal-average ensemble'] = np.mean(list(scores.values()), axis=0)
    predictions = []
    for name, score in scores.items():
        frame = base.copy()
        frame['score'] = score
        frame['predicted_probability'] = score
        frame = attach_prediction_labels(frame, orders, lookback=LOOKBACK,
                                         prediction_threshold=0.5)
        frame['model'] = name
        frame['run_date'] = run_date
        predictions.append(frame.drop(columns=['predicted_probability']))
    return pd.concat(predictions, ignore_index=True)

# All three months were previously inspected, so they are development evidence.
RUN_DATES = ['2018-04-02', '2018-05-02', '2018-06-02']
monthly = pd.concat([monthly_predictions(day) for day in RUN_DATES], ignore_index=True)
assert not monthly.duplicated(['run_date', 'model', 'order_id']).any()
print(f'{monthly["order_id"].nunique():,} distinct scored orders')

19,963 distinct scored orders


## 7. Compare months and daily capacity

Precision and recall use the provisional 0.5 threshold. The 10% capacity is an illustrative contact budget, not a chosen business requirement. Coverage is the share of known late orders selected in the daily top 10%; lift compares that capture with random daily selection at the same capacity. Report label availability beside it.

In [10]:
monthly_results = []
for (run_date, name), group in monthly.groupby(['run_date', 'model'], sort=True):
    known = group.loc[group['actual_label'].notna()]
    y = known['actual_label'].astype(int)
    if known.empty:
        continue
    result = classification_metrics(y, (known['score'] >= 0.5).astype(int), known['score'])
    result.update(extra_metrics(group, fraction=0.10))
    monthly_results.append({'month': run_date[:7], 'model': name, **result})
monthly_results = pd.DataFrame(monthly_results)
display(monthly_results[['month', 'model', 'pr_auc', 'brier_score', 'precision',
                         'recall', 'f1_score', 'coverage', 'lift',
                         'label_availability']].round(3))
print('Mean across months (each month has equal weight):')
display(monthly_results.groupby('model')[['pr_auc', 'brier_score', 'precision',
                                           'recall', 'f1_score', 'coverage', 'lift',
                                           'label_availability']].mean().round(3))

,month,model,pr_auc,brier_score,precision,recall,f1_score,coverage,lift,label_availability
0,2018-04,CatBoost,0.103,0.250,0.080,0.515,0.139,0.160,1.562,1.000
1,2018-04,Equal-average ensemble,0.121,0.233,0.103,0.609,0.176,0.202,1.980,1.000
2,2018-04,LightGBM,0.119,0.251,0.094,0.647,0.165,0.178,1.738,1.000
3,2018-04,Logistic Regression,0.117,0.261,0.108,0.690,0.186,0.213,2.090,1.000
4,2018-04,Random Forest,0.134,0.193,0.135,0.481,0.210,0.207,2.024,1.000
5,2018-04,XGBoost,0.112,0.263,0.092,0.665,0.161,0.187,1.826,1.000
6,2018-05,CatBoost,0.159,0.329,0.093,0.686,0.165,0.178,1.749,1.000
7,2018-05,Equal-average ensemble,0.155,0.313,0.094,0.729,0.167,0.176,1.732,1.000
8,2018-05,LightGBM,0.149,0.352,0.095,0.793,0.169,0.183,1.800,1.000
9,2018-05,Logistic Regression,0.121,0.282,0.107,0.615,0.182,0.169,1.664,1.000


Mean across months (each month has equal weight):


,pr_auc,brier_score,precision,recall,f1_score,coverage,lift,label_availability
model,,,,,,,,
CatBoost,0.118,0.227,0.074,0.473,0.127,0.190,1.868,0.999
Equal-average ensemble,0.124,0.227,0.082,0.555,0.142,0.219,2.145,0.999
LightGBM,0.122,0.236,0.082,0.568,0.142,0.205,2.016,0.999
Logistic Regression,0.092,0.263,0.083,0.632,0.145,0.201,1.968,0.999
Random Forest,0.133,0.221,0.092,0.541,0.154,0.205,2.012,0.999
XGBoost,0.119,0.239,0.084,0.553,0.144,0.212,2.078,0.999


## 8. Historical performance feature check

Seller and product late rates use only outcomes already known before each order’s approval day. A new entity receives the overall late rate known on that day, and its no-history share records the missing history. Compare the original features with the added history on the same monthly cohorts and fitting split. Orders in a scored month are excluded from the source of historical outcomes for that month. These are development results; an improvement here would still need confirmation on a final untouched period.


In [11]:
history_rows = []
for run_date in RUN_DATES:
    month = pd.Period(run_date, freq='M')
    month_ids = base_features.loc[
        base_features['order_approved_dt'].dt.to_period('M').eq(month), 'order_id']
    month_features = add_historical_performance(
        base_features, olist, exclude_order_ids=month_ids)
    _, history = training_window(run_date, month_features)
    fit, _, y_fit, _ = train_test_split(
        history, history['late'], test_size=30/PXD,
        random_state=RANDOM_STATE, stratify=history['late'])
    cohort = month_features.loc[
        month_features['order_approved_dt'].dt.to_period('M').eq(month)].copy()
    delivered_day = cohort['order_delivered_customer_date'].dt.normalize()
    cohort = cohort.loc[delivered_day.isna() |
                        delivered_day.gt(cohort['order_approved_dt'])].copy()
    for group, cols in [
        ('Base', NUM_COLS),
        ('Base + seller history', NUM_COLS + [c for c in HISTORY_NUM_COLS if c.startswith('seller_')]),
        ('Base + product history', NUM_COLS + [c for c in HISTORY_NUM_COLS if c.startswith('product_')]),
        ('Base + both histories', NUM_COLS + HISTORY_NUM_COLS),
    ]:
        selected = cols + CAT_COLS
        prep = ColumnTransformer([
            ('numeric', SimpleImputer(strategy='median'), cols),
            ('categorical', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
        ], sparse_threshold=0)
        model = Pipeline([('preprocessor', prep),
                          ('classifier', LGBMClassifier(
                              scale_pos_weight=y_fit.eq(0).sum() / y_fit.eq(1).sum(),
                              random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1))])
        model.fit(fit[selected], y_fit)
        scores = model.predict_proba(cohort[selected])[:, 1]
        scored = pd.DataFrame({'order_id': cohort['order_id'],
                               'predicted_probability': scores})
        scored = attach_prediction_labels(scored, orders, lookback=LOOKBACK)
        known = scored['actual_label'].notna()
        result = classification_metrics(scored.loc[known, 'actual_label'].astype(int),
                                        scored.loc[known, 'predicted_label'],
                                        scored.loc[known, 'predicted_probability'])
        history_rows.append({'month': run_date[:7], 'features': group,
                             'average_precision': result['pr_auc'],
                             'known_orders': int(known.sum())})
history_comparison = pd.DataFrame(history_rows)
display(history_comparison.round(3))
print('Equal-month mean average precision:')
display(history_comparison.groupby('features')['average_precision'].mean().round(3))


,month,features,average_precision,known_orders
0,2018-04,Base,0.129,6732
1,2018-04,Base + seller history,0.129,6732
2,2018-04,Base + product history,0.113,6732
3,2018-04,Base + both histories,0.119,6732
4,2018-05,Base,0.139,7065
5,2018-05,Base + seller history,0.149,7065
6,2018-05,Base + product history,0.148,7065
7,2018-05,Base + both histories,0.149,7065
8,2018-06,Base,0.100,6155
9,2018-06,Base + seller history,0.095,6155


Equal-month mean average precision:


features
Base                      0.123
Base + both histories     0.122
Base + product history    0.119
Base + seller history     0.124
Name: average_precision, dtype: float64

## 9. Paired model comparison

Each candidate is compared with the Logistic Regression baseline on the same orders. Resampling keeps whole dates together in seven-day blocks within each month, then averages the monthly AP differences. These descriptive intervals cover the development months and the fitted models; they do not include training variability or establish performance in a new month. No model is selected from these intervals alone.

In [12]:
comparison_rows = []
for candidate in ['XGBoost', 'Random Forest', 'CatBoost',
                  'LightGBM', 'Equal-average ensemble']:
    comparison_rows.append(paired_monthly_ap(monthly, candidate, repeats=499,
                                              block_days=7, seed=RANDOM_STATE))
display(pd.DataFrame(comparison_rows).round(3))

,candidate,reference,mean_ap_difference,ci_low,ci_high,months,block_days
0,XGBoost,Logistic Regression,0.027,0.009,0.051,3,7
1,Random Forest,Logistic Regression,0.041,0.021,0.062,3,7
2,CatBoost,Logistic Regression,0.026,0.004,0.049,3,7
3,LightGBM,Logistic Regression,0.029,0.011,0.051,3,7
4,Equal-average ensemble,Logistic Regression,0.032,0.014,0.054,3,7


## 10. Interpretation and limitations

The `pr_auc` column reports average precision. Compare monthly results and their variation before choosing a primary metric. The equal-average ensemble combines the five classifier scores with fixed weights; its outputs are not assumed calibrated probabilities.

The four historical rate and no-history measures are compared with the original predictors across the development months. All classifiers use them, and the original 31-feature set remains a benchmark. The backtest months are development evidence. The analysis has not tuned thresholds, calibrated probabilities, or run an independent final test. Historical availability of some related-table features is an assumption because the data do not record revisions to those tables.

## References

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1-3. https://doi.org/10.1175/1520-0493(1950)078<0001:VOFEIT>2.0.CO;2
- Davis, J., & Goadrich, M. (2006). The relationship between Precision-Recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233-240. https://doi.org/10.1145/1143844.1143874
- Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432
- Bergmeir, C., Hyndman, R. J., & Koo, B. (2018). A note on the validity of cross-validation for evaluating autoregressive time series prediction. *Data Mining and Knowledge Discovery*, 32, 1614-1630. https://doi.org/10.1007/s10618-018-0589-0
- Kunsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics*, 17(3), 1217-1241. https://doi.org/10.1214/aos/1176347265